# Ariane - Full Comparison Matrix (baselines in separate cells)

**2 LLM methods x 2 baselines x {text, image}**, plus greedy + random-search controls.

| | weak baseline (area, ~1.25e5) | strong baseline (topology, ~8.4e4) |
|---|---|---|
| **LLM region** | text / image | text / image |
| **LLM ordering** | text / image | text / image |

The weak and strong baselines run in **separate cells** (Cell 4 and Cell 5) so you can run/re-run each on its own. Judge LLM *value* by whether the LLM rows beat the **random-search control**, not just the baseline.

## Cell 1 - Setup (clone repo, gym shim, protobuf, anthropic)
Needs `strong_search.py` and `matrix_eval.py` pushed to the repo.

In [5]:
import subprocess, sys, os, shutil, types, glob
import numpy as np

REPO_URL  = "https://github.com/dennis5727/arianePlacement.git"
CLONE_DIR = "/kaggle/working/arianePlacement"
WORK_DIR  = os.path.join(CLONE_DIR, "maskplace")

if not os.path.exists(CLONE_DIR):
    try:
        subprocess.check_call(["git","clone","--depth","1",REPO_URL,CLONE_DIR]); print("cloned")
    except Exception as e: print("git clone failed:", e)
else:
    subprocess.call(["git","-C",CLONE_DIR,"pull","--ff-only"]); print("pulled latest")

if not os.path.exists(WORK_DIR):
    hits=glob.glob("/kaggle/input/**/place_db.py",recursive=True)
    assert hits,"no code via git or dataset"
    shutil.copytree(os.path.dirname(hits[0]),WORK_DIR); print("dataset fallback")

os.chdir(WORK_DIR); sys.path.insert(0,WORK_DIR); print("cwd:",os.getcwd())

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION","python")
try: subprocess.check_call([sys.executable,"-m","pip","install","-q","protobuf==3.20.3"])
except subprocess.CalledProcessError as e: print("protobuf pin failed:",e)

try:
    import gym; print("real gym:",gym.__version__)
except Exception:
    gym=types.ModuleType("gym"); spaces=types.ModuleType("gym.spaces")
    class _E: pass
    class _D:
        def __init__(s,n): s.n=int(n)
        def contains(s,x):
            try: x=int(x)
            except: return False
            return 0<=x<s.n
    class _B:
        def __init__(s,low=0,high=1,shape=None,dtype=None): s.low,s.high,s.shape,s.dtype=low,high,shape,dtype
    gym.Env=_E; spaces.Discrete=_D; spaces.Box=_B; gym.spaces=spaces
    sys.modules["gym"]=gym; sys.modules["gym.spaces"]=spaces; print("gym shim")

try: subprocess.check_call([sys.executable,"-m","pip","install","-q","anthropic"])
except subprocess.CalledProcessError as e: print("anthropic skipped:",e)

need=["place_db.py","place_env/place_env.py","comp_res.py","greedy_place.py",
      "region_constraint.py","parse_netlist.py","visualize.py","history_tracker.py",
      "llm_interface.py","llm_guided_placement.py","strong_search.py","matrix_eval.py",
      "ariane/netlist.pb.txt"]
miss=[f for f in need if not os.path.exists(f)]
assert not miss, f"MISSING (push to repo): {miss}"
print("files OK\n=== SETUP COMPLETE ===")

From https://github.com/dennis5727/arianePlacement
   cf7599b..5533fd4  main       -> origin/main


Updating cf7599b..5533fd4
Fast-forward
 .DS_Store                | Bin 6148 -> 6148 bytes
 ariane_full_matrix.ipynb |  70 +++++++++++++++++++++++------
 maskplace/matrix_eval.py | 113 +++++++++++++++++++++++++++++------------------
 3 files changed, 126 insertions(+), 57 deletions(-)
pulled latest
cwd: /kaggle/working/arianePlacement/maskplace
real gym: 0.26.2
files OK
=== SETUP COMPLETE ===


## Cell 2 - Sanity check

In [6]:
from place_db import PlaceDB
placedb=PlaceDB("ariane")
hard=sum(1 for n in placedb.node_info if placedb.node_info[n].get("is_hard"))
print("Nodes",len(placedb.node_info),"| Nets",len(placedb.net_info),
      "| Canvas",placedb.max_height,"| Hard",hard)
assert placedb.max_height==357

area_sum = 99908.97641387026
pin_cnt = 22802
adjust net size = 12404
node_net_num_max 220
node_area_max = 565.3772979022979
Nodes 932 | Nets 12404 | Canvas 357 | Hard 133


## API key (needed for the paid matrix below)

In [7]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"]=UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
print("API key loaded")

API key loaded


## Cell 3 - Prepare baselines (run once)
Builds both baseline orders; set MODEL / MAX_ITERS here.

In [8]:
import importlib, matrix_eval, strong_search, llm_guided_placement
importlib.reload(strong_search); importlib.reload(llm_guided_placement); importlib.reload(matrix_eval)
from matrix_eval import prepare_baselines, run_baseline, print_matrix, save_csv

MODEL     = "claude-sonnet-4-6"   # bump to "claude-opus-4-8" for a stronger (pricier) run
MAX_ITERS = 8                     # LLM iterations per cell
N_RANDOM  = 12                    # no-LLM random-search budget per baseline
GRID      = 224

# build BOTH baseline orders once; run each in its own cell below
placedb, ORDERS = prepare_baselines("ariane")
print("baselines:", {k: len(v) for k,v in ORDERS.items()})

area_sum = 99908.97641387026
pin_cnt = 22802
adjust net size = 12404
node_net_num_max 220
node_area_max = 565.3772979022979
baselines: {'weak (area)': 133, 'strong (topo)': 133}


## Cell 4 - WEAK baseline (PAID: 4 LLM runs)

In [9]:
# ---- WEAK baseline (area order, ~1.25e5): region/order x text/image + controls ----
bname = "weak (area)"
rows_weak = run_baseline(placedb, bname, ORDERS[bname], benchmark="ariane", grid=GRID,
                         model=MODEL, max_iters=MAX_ITERS, patience=3, advisor="claude",
                         n_random=N_RANDOM, outdir="/kaggle/working")
print_matrix(rows_weak)


===== baseline: weak (area) (133 macros) =====
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
  greedy baseline HPWL = 1.2549e+05
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.net_cnt 12404
self.ratio = 1.59
grid * grid 50176
placedb.node_cnt 932
placedb.

## Cell 5 - STRONG baseline (PAID: 2 LLM runs — ordering text+image only)
Region runs are skipped here (`do_region=False`) to save API cost.

In [ ]:
# ---- STRONG baseline (topology order, ~8.4e4): ordering x text/image + controls ----
# region runs skipped here (do_region=False) to save API cost; ordering is the lever
# with real headroom on the strong baseline.
bname = "strong (topo)"
rows_strong = run_baseline(placedb, bname, ORDERS[bname], benchmark="ariane", grid=GRID,
                           model=MODEL, max_iters=MAX_ITERS, patience=3, advisor="claude",
                           n_random=N_RANDOM, outdir="/kaggle/working", do_region=False)
print_matrix(rows_strong)

## Cell 6 - Combined results (run after Cells 4 & 5)

In [ ]:
# ---- combined table + CSV + best layouts (run after BOTH baseline cells) ----
from matrix_eval import print_matrix, save_csv
rows = rows_weak + rows_strong
print_matrix(rows)
save_csv(rows, "/kaggle/working/matrix.csv")

from IPython.display import Image, display
import os, glob
for p in sorted(glob.glob("/kaggle/working/*_region_*.png")+glob.glob("/kaggle/working/*_order_*.png")):
    print(os.path.basename(p)); display(Image(p))